# AIS Data Processing - Java (JMEOS)

This notebook demonstrates how to process AIS (Automatic Identification System) maritime data using **JMEOS**, the Java binding for the MEOS C library.

AIS data records the position, speed, and identity of ships at regular intervals. MEOS provides native types for temporal data (sequences of values over time) and spatial indexing structures. 

**Sections:**
- [1 - Reading from a CSV file](#n02)
- [2 - Assembling vessel trips](#n03)
- [3 - Storing data in MobilityDB](#n04)
- [4 - Port presence detection with RTree index](#n14)

## Step 1- Loading the JMEOS JAR

The `%jars` magic command adds a JAR to the JShell classpath. It must be the **only kind of content** of its cell since mixing it with comments or Java code causes the rapaio-jupyter-kernel to fail.

In [2]:
%jars ../../../jar/JMEOS-fat.jar

Add /workspace/src/main/java/examples/../../../jar/JMEOS-fat.jar to classpath

## Step 2 - Imports

Only JNR-FFI classes are imported here. The full `functions.functions` class from JMEOS is intentionally **not** imported: it contains roughly 12,000 lines, and JNR-FFI generates a proxy class that exceeds the JVM's method size limit, causing a `Method too large` error at load time.

Instead, we declare a minimal interface with only the functions we actually need.

In [2]:
import jnr.ffi.*;
import jnr.ffi.LibraryLoader;
import java.io.*;
import java.time.*;
import java.util.*;

System.out.println("Imports OK");

Imports OK


## Step 3 - Minimal MEOS interface

**Why a custom interface instead of using JMEOS directly?**

JNR-FFI works by generating a Java proxy that maps interface methods to native C function calls. When a native function returns a `char*` (C string), JNR-FFI automatically calls `free()` on the returned pointer after copying it to a Java `String`.

**Solution:** declare all text-returning functions with `Pointer` as the return type.

**Why `tgeompoint_in` and not `tgeogpoint_in`?**

When calling MEOS directly through JNR-FFI, the library runs standalone without PostGIS, so its SRID catalogue is absent. The geography type (tgeogpoint) requires that catalogue to validate coordinate systems and crashes with "got NULL for SRID (4326)". The geometry type (tgeompoint) has no such dependency. In exercise 3, SQL is sent to MobilityDB which runs on top of PostGIS: there the full catalogue is available and tgeogpoint with SRID=4326 works correctly.

**Error handler interface:**

MEOS requires an error handler callback to be registered before parsing functions are called. Without it, any internal MEOS error invokes a NULL function pointer, immediately crashing the JVM.

In [3]:
// The error handler is a C callback — JNR-FFI maps it to a function pointer.
interface error_handler_fn {
    @jnr.ffi.annotations.Delegate
    void call(int level, int code, String message);
}

// Minimal interface: only the functions used in this notebook.
// Return types that would be String in pure Java are declared as Pointer
// to prevent JNR-FFI from calling free() on MEOS-managed memory.
interface MeosAIS {

    void meos_initialize_timezone(String tz);
    void meos_initialize_error_handler(error_handler_fn handler);
    void meos_finalize();

    // String-based constructors: parse a WKT literal into a native MEOS object.
    // Instant format  : "POINT(lon lat)@2021-01-08 00:00:00+00"
    // Sequence format : "[POINT(lon lat)@t1, POINT(lon lat)@t2, ...]"
    Pointer tgeompoint_in(String str);
    Pointer tfloat_in(String str);

    // Output functions declared as Pointer.
    Pointer tspatial_out(Pointer temp, int maxdd);
    Pointer tfloat_out(Pointer temp, int maxdd);

    // STBox (Spatio-Temporal Box) - bounding box used by the RTree index.
    // Spatial-only format  : "STBOX X((xmin ymin),(xmax ymax))"
    // Spatio-temporal format: "STBOX XT(((xmin ymin),(xmax ymax)),(tmin,tmax))"
    Pointer stbox_in(String str);
    Pointer stbox_out(Pointer box, int maxdd);
    boolean overlaps_stbox_stbox(Pointer a, Pointer b);

    // Geometry functions used for exact point-in-circle verification.
    // geom_from_text parses a WKT string into a native GSERIALIZED geometry.
    // geom_intersects2d returns true if two 2D geometries share any point.
    Pointer geo_from_text(String wkt, int srid);
    boolean geom_intersects2d(Pointer gs1, Pointer gs2);

    // RTree spatial index functions.
    Pointer rtree_create_stbox();
    void    rtree_insert(Pointer rtree, Pointer box, int id);
    Pointer rtree_search(Pointer rtree, Pointer box, Pointer countPtr);
    void    rtree_free(Pointer rtree);

    // Metric functions.
    double tpoint_length(Pointer temp);
    double tnumber_twavg(Pointer temp);
    int    temporal_num_instants(Pointer temp);
}

// Utility function: read a C string from a Pointer without freeing it.
String ptrToString(Pointer p) {
    if (p == null) return "(null)";
    return p.getString(0);
}

System.out.println("Interface defined");

Interface defined


## Step 4 - Loading libmeos.so and initialising MEOS

JNR-FFI locates the native library by searching the given directory for a file named `libmeos.so` (Linux). This notebook requires the Linux version compiled from source - see the setup guide.

Initialisation must happen in two separate calls:
1. `meos_initialize_timezone` sets the time zone used when parsing timestamps.
2. `meos_initialize_error_handler` registers the callback. This **must** be called before any parsing function, otherwise internal errors crash the JVM.

In [4]:
String MEOS_LIB_PATH = "/workspace/src";

MeosAIS meos = LibraryLoader.create(MeosAIS.class)
    .search(MEOS_LIB_PATH)
    .load("meos");

// Lambda that prints any MEOS error instead of letting it crash the JVM.
error_handler_fn errorHandler = (level, code, msg) ->
    System.err.println("MEOS [" + level + "/" + code + "]: " + msg);

meos.meos_initialize_timezone("UTC");
meos.meos_initialize_error_handler(errorHandler);
System.out.println("MEOS initialised");

MEOS initialised


---
## <a id="n02"></a> 1 - Reading from a CSV file

Source: `ais_instants.csv`

```
t,mmsi,latitude,longitude,sog
2021-01-08 00:00:00,265513270,57.059,12.272388,0
2021-01-08 00:00:01,219027804,55.94244,11.866278,0
...
```

Each row is one AIS observation: a ship identified by its MMSI number, at a given position (latitude, longitude) and speed (SOG), recorded at a specific timestamp.

### 1.1 File verification

In [5]:
String INPUT_FILE = "/workspace/src/main/java/examples/data/ais_instants.csv";

var csvFile = new java.io.File(INPUT_FILE);
if (!csvFile.exists()) {
    System.err.println("File not found: " + INPUT_FILE);
    System.err.println("Search with: find /workspace -name ais_instants.csv");
} else {
    System.out.println("File found (" + csvFile.length() + " bytes)");
}

File found (8311129 bytes)


### 1.2 Reading and displaying sample instants

For each row we build two temporal objects:
- a `tgeompoint` instant from `"POINT(lon lat)@timestamp"`
- a `tfloat` instant from `"value@timestamp"`

Note the use of `Locale.US` in `String.format` just in case systems configured with a non-English locale could produce `"POINT(12,272388 57,059)"` with a comma as the decimal separator, which MEOS cannot parse.

In [6]:
int noRecords = 0;
int noNulls   = 0;

try (var reader = new BufferedReader(new FileReader(INPUT_FILE))) {
    reader.readLine(); // skip header

    String line;
    while ((line = reader.readLine()) != null) {
        String[] f = line.split(",");
        if (f.length != 5) { noNulls++; continue; }

        try {
            String ts  = f[0].trim();
            long  mmsi = Long.parseLong(f[1].trim());
            double lat = Double.parseDouble(f[2].trim());
            double lon = Double.parseDouble(f[3].trim());
            double sog = Double.parseDouble(f[4].trim());
            noRecords++;

            if (noRecords % 1000 == 0) {
                Pointer locInst = meos.tgeompoint_in(
                    String.format(java.util.Locale.US,
                        "POINT(%f %f)@%s", lon, lat, ts));
                Pointer sogInst = meos.tfloat_in(
                    String.format(java.util.Locale.US, "%f@%s", sog, ts));
                System.out.printf("[%6d] MMSI: %d | Pos: %s | SOG: %s%n",
                    noRecords, mmsi,
                    ptrToString(meos.tspatial_out(locInst, 4)),
                    ptrToString(meos.tfloat_out(sogInst, 2)));
            }
        } catch (NumberFormatException e) { noNulls++; }
    }
}

System.out.printf("%nSummary%n");
System.out.printf("  Records read   : %,d%n", noRecords);
System.out.printf("  Records skipped: %,d%n", noNulls);

[  1000] MMSI: 566948000 | Pos: 01010000007F30F0DC7B781240E5266A696EC94B40@2021-01-08 00:10:46+00 | SOG: 0.5@2021-01-08 00:10:46+00
[  2000] MMSI: 566948000 | Pos: 01010000008527F4FA93781240DD7C23BA67C94B40@2021-01-08 00:21:25+00 | SOG: 0.1@2021-01-08 00:21:25+00
[  3000] MMSI: 219001559 | Pos: 01010000005DBF60376CF32340F645425BCECB4C40@2021-01-08 00:33:48+00 | SOG: 0.1@2021-01-08 00:33:48+00
[  4000] MMSI: 257136000 | Pos: 010100000067B5C01E13891D402BA391CF2B804C40@2021-01-08 00:45:54+00 | SOG: 14.2@2021-01-08 00:45:54+00
[  5000] MMSI: 219027804 | Pos: 01010000005A29047289BB27404E9B711AA2F84B40@2021-01-08 00:56:55+00 | SOG: 0@2021-01-08 00:56:55+00
[  6000] MMSI: 566948000 | Pos: 0101000000BE840A0E2F781240CC96AC8A70C94B40@2021-01-08 01:06:01+00 | SOG: 0.4@2021-01-08 01:06:01+00
[  7000] MMSI: 219027804 | Pos: 0101000000965B5A0D89BB27404E9B711AA2F84B40@2021-01-08 01:13:57+00 | SOG: 0@2021-01-08 01:13:57+00
[  8000] MMSI: 566948000 | Pos: 010100000086E63A8DB4741240B03C484F91C94B40@2021

java.io.PrintStream@121d9fab

---
## <a id="n03"></a> 2 - Assembling vessel trips

A "trip" is a temporal sequence of positions for a single ship. The strategy here is to accumulate WKT strings per vessel in memory, then pass the complete sequence string to MEOS at the end.

This is simpler than calling native instant constructors one by one, and avoids the risk of passing malformed native pointers between calls.

### 2.1 Trip data structure

Each `TripRecord` holds two lists of WKT strings (one for position, one for SOG). The `buildTripSeq` method joins them into a MEOS sequence literal:
`"[POINT(12.27 57.05)@2021-01-08 00:00:00+00, POINT(12.28 57.06)@2021-01-08 00:01:00+00, ...]"`

In [7]:
class TripRecord {
    long         mmsi;
    List<String> tripWkt = new ArrayList<>(); // WKT instants for position
    List<String> sogWkt  = new ArrayList<>(); // WKT instants for speed

    TripRecord(long mmsi) { this.mmsi = mmsi; }

    // Builds a tgeompoint sequence from the accumulated WKT instant strings.
    Pointer buildTripSeq() {
        return meos.tgeompoint_in("[" + String.join(", ", tripWkt) + "]");
    }

    // Builds a tfloat sequence from the accumulated WKT instant strings.
    Pointer buildSogSeq() {
        return meos.tfloat_in("[" + String.join(", ", sogWkt) + "]");
    }
}

### 2.2 Reading and assembling

In [8]:
// Limit to the first MAX_SHIPS vessels to keep execution time short.
int MAX_SHIPS = 5;

Map<Long, TripRecord> trips = new LinkedHashMap<>();
noRecords = 0; noNulls = 0;
long t0 = System.currentTimeMillis();

try (var reader = new BufferedReader(new FileReader(INPUT_FILE))) {
    reader.readLine(); // skip header

    String line;
    while ((line = reader.readLine()) != null) {
        String[] f = line.split(",");
        if (f.length != 5) { noNulls++; continue; }

        try {
            String ts  = f[0].trim();
            long  mmsi = Long.parseLong(f[1].trim());
            double lat = Double.parseDouble(f[2].trim());
            double lon = Double.parseDouble(f[3].trim());
            double sog = Double.parseDouble(f[4].trim());
            noRecords++;

            // computeIfAbsent creates a new TripRecord only if the MMSI
            // has not been seen before, and only if the ship limit is not reached.
            TripRecord trip = trips.computeIfAbsent(mmsi,
                k -> trips.size() < MAX_SHIPS ? new TripRecord(k) : null);
            if (trip == null) continue;

            trip.tripWkt.add(String.format(java.util.Locale.US,
                "POINT(%f %f)@%s", lon, lat, ts));
            trip.sogWkt.add(String.format(java.util.Locale.US,
                "%f@%s", sog, ts));

        } catch (NumberFormatException e) { noNulls++; }
    }
}

System.out.printf("Read: %,d records, %d skipped, %d vessels%n",
    noRecords, noNulls, trips.size());

Read: 156,837 records, 0 skipped, 5 vessels


java.io.PrintStream@38045cf1

### 2.3 Building sequences

- `tpoint_length` returns the total distance travelled in metres.
- `tnumber_twavg` returns the time-weighted average: instead of weighting all instants equally, it weights each value by the duration of the interval during which it was valid. This gives a more accurate average for unevenly sampled data.

In [9]:
System.out.println("+----------------+--------------+------------------+--------------------+");
System.out.println("|      MMSI      |  # Instants  |   Distance (m)   |    Avg Speed TWA   |");
System.out.println("+----------------+--------------+------------------+--------------------+");

for (TripRecord tr : trips.values()) {
    Pointer tripSeq = tr.buildTripSeq();
    Pointer sogSeq  = tr.buildSogSeq();
    int    nInst = meos.temporal_num_instants(tripSeq);
    double dist  = meos.tpoint_length(tripSeq);
    double twavg = meos.tnumber_twavg(sogSeq);
    System.out.printf("| %14d | %12d | %16.3f | %18.3f |%n",
        tr.mmsi, nInst, dist, twavg);
}
System.out.println("+----------------+--------------+------------------+--------------------+");
System.out.printf("%nElapsed: %.2f s%n", (System.currentTimeMillis() - t0) / 1000.0);

+----------------+--------------+------------------+--------------------+
|      MMSI      |  # Instants  |   Distance (m)   |    Avg Speed TWA   |
+----------------+--------------+------------------+--------------------+
|      265513270 |         7968 |            0.015 |              0.057 |
|      219027804 |        17152 |            0.862 |              1.446 |
|      566948000 |        24269 |            0.255 |              0.305 |
|      219001559 |        36133 |            0.161 |              0.030 |
|      257136000 |        21090 |            8.536 |             14.589 |
+----------------+--------------+------------------+--------------------+

Elapsed: 3.45 s


java.io.PrintStream@3928e193

### 2.4 Exporting assembled trips to CSV

In [10]:
String OUTPUT_FILE = "/workspace/src/main/java/examples/data/ais_trips_assembled.csv";

try (var writer = new BufferedWriter(new FileWriter(OUTPUT_FILE))) {
    writer.write("mmsi,trip,sog\n");
    for (TripRecord tr : trips.values()) {
        Pointer tripSeq = tr.buildTripSeq();
        Pointer sogSeq  = tr.buildSogSeq();
        writer.write(String.format("%d,%s,%s%n",
            tr.mmsi,
            ptrToString(meos.tspatial_out(tripSeq, 6)),
            ptrToString(meos.tfloat_out(sogSeq, 6))));
    }
}
System.out.println("File exported: " + OUTPUT_FILE);

File exported: /workspace/src/main/java/examples/data/ais_trips_assembled.csv


---
## <a id="n04"></a> 3 - Storing data in MobilityDB

MobilityDB is a PostgreSQL extension that adds native temporal types such as `tgeogpoint` and `tfloat`. Data is inserted via standard JDBC using SQL literals in the MobilityDB WKT format.

**Prerequisite:** start a MobilityDB container in a separate terminal:
```bash
docker run --name mobilitydb -e POSTGRES_PASSWORD=postgres \
  -p 5432:5432 -d mobilitydb/mobilitydb
```

### 3.1 Loading the PostgreSQL JDBC driver

The JDBC driver is a JAR that implements the PostgreSQL wire protocol. Without it, `DriverManager.getConnection(...)` throws `No suitable driver found for jdbc:postgresql://...`.

Download `postgresql-42.7.10.jar` from [jdbc.postgresql.org](https://jdbc.postgresql.org/download/) and place it in `/workspace/jar/`.

In [11]:
%jars ../../../../jar/postgresql-42.7.10.jar

Add /workspace/src/main/java/examples/../../../../jar/postgresql-42.7.10.jar to classpath

In [12]:
import java.sql.*;
System.out.println("JDBC driver loaded");

JDBC driver loaded


Note how the %jars magic command and the java import code cell are separated to avoid mixing them and making the rapaio-jupyter-kernel for Java fail

### 3.2 Connecting to the database

On Windows and macOS, Docker Desktop maps `host.docker.internal` to the host machine, which is where the MobilityDB container exposes port 5432. On Linux, replace this with the container's IP address, or use a Docker network (cf. the comments in the N04_AIS_Store program).

In [13]:
String JDBC_URL = "jdbc:postgresql://host.docker.internal:5432/postgres"
    + "?user=postgres&password=postgres";

Connection conn = DriverManager.getConnection(JDBC_URL);
// Setting an empty search_path prevents untrusted users from hijacking
// function lookups by placing objects in a schema that is searched first.
try (Statement s = conn.createStatement()) {
    s.execute("SET search_path = ''");
}
System.out.println("Connected to MobilityDB");

Connected to MobilityDB


### 3.3 Creating extensions and table

In [14]:
try (Statement s = conn.createStatement()) {
    s.execute("CREATE EXTENSION IF NOT EXISTS postgis");
    s.execute("CREATE EXTENSION IF NOT EXISTS mobilitydb");
    System.out.println("Extensions PostGIS + MobilityDB ready");

    s.execute("DROP TABLE IF EXISTS public.AISInstants");
    s.execute(
        "CREATE TABLE public.AISInstants(" +
        "MMSI integer, " +
        "location public.tgeogpoint, " +
        "SOG public.tfloat)");
    System.out.println("Table AISInstants created");
}

Extensions PostGIS + MobilityDB ready
Table AISInstants created


### 3.4 Bulk insert

Sending one `INSERT` per row would be extremely slow due to the round-trip latency between the Java process and the database. Instead, we accumulate `NO_BULK_INSERT` rows into a single multi-value `INSERT` statement and send them in one batch.

MobilityDB accepts temporal instant literals directly in SQL:
`'SRID=4326;Point(12.27 57.05)@2021-01-08 00:00:00+00'`

The transaction is committed once after all inserts, reducing disk access and improving throughput.

In [15]:
int NO_INSTS_BATCH = 10000; // print a marker every N records
int NO_BULK_INSERT = 20;    // rows per INSERT statement

conn.setAutoCommit(false);

int noStore = 0, noStoreNulls = 0, batchCount = 0;
var insertBuffer = new StringBuilder();
long t0store = System.currentTimeMillis();

System.out.printf("Inserting (one '*' every %d records)%n", NO_INSTS_BATCH);

try (var reader = new java.io.BufferedReader(new java.io.FileReader(INPUT_FILE))) {
    reader.readLine(); // skip header

    String line;
    while ((line = reader.readLine()) != null) {
        String[] f = line.split(",");
        if (f.length != 5) { noStoreNulls++; continue; }

        try {
            String ts  = f[0].trim();
            long  mmsi = Long.parseLong(f[1].trim());
            double lat = Double.parseDouble(f[2].trim());
            double lon = Double.parseDouble(f[3].trim());
            double sog = Double.parseDouble(f[4].trim());
            noStore++;

            if (noStore % NO_INSTS_BATCH == 0) {
                System.out.print("*"); System.out.flush();
            }

            if (batchCount == 0) {
                insertBuffer.setLength(0);
                insertBuffer.append(
                    "INSERT INTO public.AISInstants(MMSI, location, SOG) VALUES ");
            }

            insertBuffer.append(String.format(java.util.Locale.US,
                "(%d, 'SRID=4326;Point(%f %f)@%s', '%f@%s'),",
                mmsi, lon, lat, ts, sog, ts));
            batchCount++;

            if (batchCount == NO_BULK_INSERT) {
                insertBuffer.setLength(insertBuffer.length() - 1); // remove trailing comma
                insertBuffer.append(";");
                try (Statement s = conn.createStatement()) {
                    s.execute(insertBuffer.toString());
                }
                batchCount = 0;
            }
        } catch (NumberFormatException e) { noStoreNulls++; }
    }
}

// Send the last incomplete batch if any.
if (batchCount > 0) {
    insertBuffer.setLength(insertBuffer.length() - 1);
    insertBuffer.append(";");
    try (Statement s = conn.createStatement()) {
        s.execute(insertBuffer.toString());
    }
}

conn.commit();
System.out.printf("%n%n%,d records inserted, %d skipped in %.2f s%n",
    noStore, noStoreNulls, (System.currentTimeMillis() - t0store) / 1000.0);

Inserting (one '*' every 10000 records)
***************

156,837 records inserted, 0 skipped in 11.00 s


java.io.PrintStream@2be9431f

### 3.5 Verification

In [16]:
try (Statement s = conn.createStatement();
     ResultSet rs = s.executeQuery(
         "SELECT COUNT(*) FROM public.AISInstants")) {
    rs.next();
    System.out.printf("Rows in AISInstants: %d%n", rs.getInt(1));
}

Rows in AISInstants: 156837


### 3.6 Cleanup

In [17]:
if (conn != null && !conn.isClosed()) {
    conn.close();
    System.out.println("Connection closed");
}
meos.meos_finalize();
System.out.println("MEOS finalised");

Connection closed
MEOS finalised


---
## <a id="n14"></a> 4 - Port presence detection with RTree index

**Objective:** for each AIS position, determine whether the ship is within the boundary of a port, and record when.

The exercise compares two approaches:

| | Brute force | RTree index |
|---|---|---|
| Method | Check every port for every position | Index the ports; query returns only candidates |
| Complexity | O(positions x ports) | O(positions x log ports) |
| When it wins | Very few ports | Many ports or many queries |

**Files:** `ais_instants.csv` + `port.csv`

### 4.1 Utility functions: WKB decoding and coordinate conversion

The `port.csv` file stores each port's geometry as EWKB (Extended Well-Known Binary), a compact binary format used by PostgreSQL/PostGIS, encoded as a hexadecimal string.

`parsePortWkb` extracts the circle centre by averaging the 32 perimeter vertices, and computes the radius as the distance from that centre to the first vertex.

`utm32n_to_wgs84` converts UTM easting/northing (metres) to WGS84 longitude/latitude (degrees) using the inverse Transverse Mercator projection (Bowring's method). This is necessary because the AIS coordinates are in WGS84, while the port geometries are in UTM.

**Note:** this is a temporary fix, in the end it should be replaced by methods such as `geom_from_hexewkb`, `geo_transform` and `geo_to_stbox`. The problem is the lack of the PostGIS **spatial_ref_sys** catalogue which could be transformed into a csv and then setup using `meos_set_spatial_ref_sys_csv`.

In [18]:
// Parse an EWKB hex string representing a circular port polygon.
// Returns double[]{centre_easting_m, centre_northing_m, radius_m}.
double[] parsePortWkb(String hexStr) {
    byte[] b = new byte[hexStr.length() / 2];
    for (int i = 0; i < b.length; i++)
        b[i] = (byte) Integer.parseInt(hexStr.substring(2*i, 2*i+2), 16);
    // The first byte is 0x01 (little-endian) or 0x00 (big-endian).
    var buf = java.nio.ByteBuffer.wrap(b)
        .order(b[0] == 1 ? java.nio.ByteOrder.LITTLE_ENDIAN
                         : java.nio.ByteOrder.BIG_ENDIAN);
    buf.get(); // consume byte-order byte
    int gtype = buf.getInt(); // geometry type + flags
    if ((gtype & 0x20000000) != 0) buf.getInt(); // skip SRID if present
    buf.getInt(); // number of rings (always 1 for a simple polygon)
    int nPts = buf.getInt(); // total points including the closing duplicate
    // Read the nPts-1 unique vertices (the last point repeats the first).
    double[] xs = new double[nPts-1], ys = new double[nPts-1];
    for (int i = 0; i < nPts-1; i++) {
        xs[i] = buf.getDouble();
        ys[i] = buf.getDouble();
    }
    double cx = java.util.Arrays.stream(xs).average().getAsDouble();
    double cy = java.util.Arrays.stream(ys).average().getAsDouble();
    double r  = Math.sqrt(Math.pow(xs[0]-cx, 2) + Math.pow(ys[0]-cy, 2));
    return new double[]{cx, cy, r};
}

// Convert UTM zone 32N (ETRS89, SRID 25832) to WGS84 longitude/latitude.
// Uses the inverse Transverse Mercator projection (Bowring's method).
// Returns double[]{longitude_deg, latitude_deg}.
double[] utm32n_to_wgs84(double E, double N) {
    // WGS84 ellipsoid parameters.
    final double a  = 6378137;           // semi-major axis (m)
    final double f  = 1.0 / 298.257223563;
    final double b  = a * (1 - f);       // semi-minor axis
    final double e2 = 1 - (b/a)*(b/a);  // eccentricity squared
    final double ep2 = e2 / (1 - e2);   // second eccentricity squared
    final double k0 = 0.9996;            // UTM scale factor
    // Zone 32 central meridian is 9 degrees East.
    double lon0 = Math.toRadians((32 - 1) * 6 - 180 + 3);
    double x = E - 500000; // remove false easting
    double y = N;
    double M   = y / k0;
    double mu  = M / (a * (1 - e2/4 - 3*e2*e2/64 - 5*e2*e2*e2/256));
    double e1  = (1 - Math.sqrt(1 - e2)) / (1 + Math.sqrt(1 - e2));
    // Footprint latitude (phi1) — the latitude of the point on the central meridian
    // at the same meridional arc distance as the point being converted.
    double phi1 = mu
        + (3*e1/2   - 27*e1*e1*e1/32)        * Math.sin(2*mu)
        + (21*e1*e1/16 - 55*e1*e1*e1*e1/32)  * Math.sin(4*mu)
        + (151*e1*e1*e1/96)                   * Math.sin(6*mu);
    double N1 = a / Math.sqrt(1 - e2 * Math.sin(phi1) * Math.sin(phi1));
    double T1 = Math.tan(phi1) * Math.tan(phi1);
    double C1 = ep2 * Math.cos(phi1) * Math.cos(phi1);
    double R1 = a * (1 - e2) / Math.pow(1 - e2 * Math.sin(phi1) * Math.sin(phi1), 1.5);
    double D  = x / (N1 * k0);
    double lat = phi1 - (N1 * Math.tan(phi1) / R1) *
        (D*D/2 - (5 + 3*T1 + 10*C1 - 4*C1*C1 - 9*ep2) * D*D*D*D / 24);
    double lon = lon0 + (D - (1 + 2*T1 + C1) * D*D*D / 6
        + (5 - 2*C1 + 28*T1 - 3*C1*C1 + 8*ep2 + 24*T1*T1) * D*D*D*D*D / 120)
        / Math.cos(phi1);
    return new double[]{Math.toDegrees(lon), Math.toDegrees(lat)};
}

System.out.println("WKB and UTM utilities defined");

WKB and UTM utilities defined


### 4.2 Loading ports from `port.csv`

For each port we create a spatial bounding box (`STBOX X`) that encloses its 1000 m proximity zone.

The UTM radius in metres is converted to degrees of longitude using the approximation `r_deg = r_m / (111320 * cos(lat))`.

In [19]:
String PORTS_FILE = "/workspace/src/main/java/examples/data/port.csv";

record PortBox(String portId, String portName, double lon, double lat,
               double radiusDeg, Pointer box, Pointer geom) {}

List<PortBox> portList = new ArrayList<>();

try (var reader = new java.io.BufferedReader(new java.io.FileReader(PORTS_FILE))) {
    reader.readLine(); // skip header
    String line;
    while ((line = reader.readLine()) != null) {
        // CSV columns: id,port_id,data_src_c,port_coor_,cntr_code,port_name,
        //              rep_mar_l,source,traffic,geom
        // Split into at most 10 tokens to avoid splitting the WKB hex string.
        String[] f = line.split(",", 10);
        if (f.length < 10) continue;
        String portId   = f[1].trim();
        String portName = f[5].trim();
        String geomHex  = f[9].trim();
        if (geomHex.isEmpty()) continue;
        try {
            double[] cr  = parsePortWkb(geomHex);
            double[] wgs = utm32n_to_wgs84(cr[0], cr[1]);
            double lon   = wgs[0], lat = wgs[1];
            double radDeg = cr[2] / (111_320.0 * Math.cos(Math.toRadians(lat)));
            String boxStr = String.format(java.util.Locale.US,
                "STBOX X((%f %f),(%f %f))",
                lon - radDeg, lat - radDeg,
                lon + radDeg, lat + radDeg);
            Pointer box = meos.stbox_in(boxStr);
            // Build the circle polygon as a MEOS geometry for exact intersection.
            // We reconstruct it as a WKT polygon with the same 32 vertices from
            // the UTM data, now expressed in WGS84 degrees.
            var sb = new StringBuilder("POLYGON((");
            int nSides = 32;
            for (int i = 0; i <= nSides; i++) {
                double angle = 2 * Math.PI * i / nSides;
                double px = lon + radDeg * Math.cos(angle);
                double py = lat + radDeg * Math.sin(angle);
                if (i > 0) sb.append(",");
                sb.append(String.format(java.util.Locale.US, "%f %f", px, py));
            }
            sb.append("))");
            Pointer geom = meos.geo_from_text(sb.toString(), 0);
            if (box != null && geom != null)
                portList.add(new PortBox(portId, portName, lon, lat, radDeg, box, geom));
        } catch (Exception e) { /* skip malformed WKB */ }
    }
}

System.out.printf("%d ports loaded%n", portList.size());
portList.stream().limit(5).forEach(p ->
    System.out.printf("  %-10s %-25s lon=%7.4f lat=%7.4f radius=%.5f deg%n",
        p.portId(), p.portName(), p.lon(), p.lat(), p.radiusDeg()));

140 ports loaded
  DKAAB      Aabenraa                  lon= 9.4275 lat=55.0417 radius=0.01568 deg
  DKAAL      Aalborg                   lon= 9.9371 lat=57.0489 radius=0.01652 deg
  DKAAR      Århus                     lon=10.2222 lat=56.1469 radius=0.01613 deg
  DKAGH      Agger Havn                lon= 8.2565 lat=56.7763 radius=0.01640 deg
  DKAGO      Agersø                    lon=11.1977 lat=55.2121 radius=0.01574 deg


### 4.3 Loading AIS positions

Each AIS observation becomes a `STBOX X` where min equals max - effectively a point. This is used as the first-pass filter with `overlaps_stbox_stbox`.

The `lon` and `lat` fields are also kept in the record. They are needed in section 4.6 to build a `POINT` geometry for the exact `geom_intersects2d` check.

In [20]:
// geom is the POINT geometry for this position, used in the exact intersection check.
record ShipPos(long mmsi, String ts, double lon, double lat, Pointer box, Pointer geom) {}
List<ShipPos> shipPositions = new ArrayList<>();

try (var reader = new java.io.BufferedReader(new java.io.FileReader(INPUT_FILE))) {
    reader.readLine(); // skip header
    String line;
    while ((line = reader.readLine()) != null) {
        String[] f = line.split(",");
        if (f.length != 5) continue;
        try {
            String ts  = f[0].trim();
            long  mmsi = Long.parseLong(f[1].trim());
            double lat = Double.parseDouble(f[2].trim());
            double lon = Double.parseDouble(f[3].trim());
            String boxStr = String.format(java.util.Locale.US,
                "STBOX X((%f %f),(%f %f))", lon, lat, lon, lat);
            Pointer box  = meos.stbox_in(boxStr);
            Pointer geom = meos.geo_from_text(
                String.format(java.util.Locale.US, "POINT(%f %f)", lon, lat), 0);
            if (box != null && geom != null)
                shipPositions.add(new ShipPos(mmsi, ts, lon, lat, box, geom));
        } catch (NumberFormatException e) {}
    }
}
System.out.printf("%,d AIS positions loaded%n", shipPositions.size());

156,837 AIS positions loaded


java.io.PrintStream@73610901

### 4.4 Approach 1 - Brute force (STBox first pass)

For every AIS position we iterate through all ports and call `overlaps_stbox_stbox`. The STBox is the bounding square of the port circle - positions in the corners of that square pass the test even though they lie outside the actual 1000 m circle. These are **false positives**. The exact check is applied in section 4.6.

In [26]:
List<String> bruteResults = new ArrayList<>();
long t0brute = System.currentTimeMillis();

for (ShipPos pos : shipPositions) {
    for (PortBox port : portList) {
        if (meos.overlaps_stbox_stbox(pos.box(), port.box())) {
            bruteResults.add(String.format("MMSI %d @ %s -> %s (%s)",
                pos.mmsi(), pos.ts(), port.portName(), port.portId()));
        }
    }
}
long bruteMs = System.currentTimeMillis() - t0brute;

System.out.println("+- BRUTE FORCE -----------------------------------------------+");
System.out.printf( "| %,d positions x %d ports = %,d comparisons%n",
    shipPositions.size(), portList.size(),
    (long) shipPositions.size() * portList.size());
System.out.printf( "| Matches found : %d%n", bruteResults.size());
System.out.printf( "| Time          : %d ms%n", bruteMs);
System.out.println("+-------------------------------------------------------------+");
bruteResults.stream().limit(20).forEach(r -> System.out.println("  " + r));

+- BRUTE FORCE -----------------------------------------------+
| 156,837 positions x 140 ports = 21,957,180 comparisons
| Matches found : 50938
| Time          : 762 ms
+-------------------------------------------------------------+
  MMSI 219001559 @ 2021-01-08 00:00:05 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:11 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:18 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:24 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:31 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:37 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:44 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:50 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:57 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:03 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:10 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:17 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:23 -> Hirt

### 4.5 Approach 2 - RTree index (STBox first pass)

An RTree organises bounding boxes in a tree where each node stores the minimum bounding box of all its children. When querying with a given box, entire subtrees whose bounding box does not overlap the query can be skipped without examining any of their contents. This reduces the number of overlap checks from linear to logarithmic in the number of indexed items.

Like the brute force approach, this step uses `overlaps_stbox_stbox` and therefore produces the same false positives. The exact check is applied in section 4.6.

**How `rtree_search` returns results:**

`rtree_search` does not return a Java collection. It writes the count of results into a native integer that we allocate with `Memory.allocate`, and returns a native array of integer IDs (the values we passed to `rtree_insert`). We then read those IDs back with `idsPtr.get(0, ids, 0, count)`.

In [27]:
// Build the index: insert each port box with its list index as the ID.
Pointer rtree = meos.rtree_create_stbox();
for (int i = 0; i < portList.size(); i++)
    meos.rtree_insert(rtree, portList.get(i).box(), i);
System.out.printf("RTree index built: %d ports indexed%n", portList.size());

List<String> rtreeResults = new ArrayList<>();
// Runtime is needed to allocate native memory for the output count parameter.
var rt = jnr.ffi.Runtime.getSystemRuntime();
long t0rtree = System.currentTimeMillis();

// Allocate a native int that rtree_search will write the result count into.
Pointer countPtr = jnr.ffi.Memory.allocate(rt, Integer.BYTES);

for (ShipPos pos : shipPositions) {    
    countPtr.putInt(0, 0);
    // idsPtr points to a native array of matching port IDs.
    Pointer idsPtr = meos.rtree_search(rtree, pos.box(), countPtr);
    int count = countPtr.getInt(0);
    if (idsPtr != null && count > 0) {
        int[] ids = new int[count];
        idsPtr.get(0, ids, 0, count); // copy native array to Java
        for (int id : ids) {
            PortBox port = portList.get(id);
            rtreeResults.add(String.format("MMSI %d @ %s -> %s (%s)",
                pos.mmsi(), pos.ts(), port.portName(), port.portId()));
        }
    }
}
long rtreeMs = System.currentTimeMillis() - t0rtree;

System.out.println("+- RTREE INDEX -----------------------------------------------+");
System.out.printf( "| Matches found : %d%n", rtreeResults.size());
System.out.printf( "| Time          : %d ms%n", rtreeMs);
System.out.println("+-------------------------------------------------------------+");
rtreeResults.stream().limit(20).forEach(r -> System.out.println("  " + r));

RTree index built: 140 ports indexed
+- RTREE INDEX -----------------------------------------------+
| Matches found : 50938
| Time          : 443 ms
+-------------------------------------------------------------+
  MMSI 219001559 @ 2021-01-08 00:00:05 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:11 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:18 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:24 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:31 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:37 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:44 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:50 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:00:57 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:03 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:10 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:17 -> Hirtshals (DKHIR)
  MMSI 219001559 @ 2021-01-08 00:01:23 -> Hirtshals (DKHIR)
  MMSI

### 4.6 Exact verification - eliminating false positives

The STBox is a square that **circumscribes** the port circle. A ship in one of the four corners passes the STBox overlap test but lies outside the actual circle:

```
    +----------+
    | X      X |    X = corner position: inside the square, outside the circle
    |    __    |
    |   /  \   |
    |  |  O |  |    O = port centre
    |   \__/   |
    | X      X |
    +----------+
```

`geom_intersects2d` tests whether two geometries share any point. Here we pass the ship `POINT` and the port circle `POLYGON`: a point inside the polygon intersects it, a point outside does not. This eliminates all false positives introduced by the square approximation.

In [28]:
// Apply the exact check to both approaches and measure each independently.

// --- Brute force + exact check ---
List<String> bruteExact = new ArrayList<>();
long t0bruteExact = System.currentTimeMillis();
for (ShipPos pos : shipPositions) {
    for (PortBox port : portList) {
        if (meos.overlaps_stbox_stbox(pos.box(), port.box())
                && meos.geom_intersects2d(pos.geom(), port.geom())) {
            bruteExact.add(String.format("MMSI %d @ %s -> %s (%s)",
                pos.mmsi(), pos.ts(), port.portName(), port.portId()));
        }
    }
}
long bruteExactMs = System.currentTimeMillis() - t0bruteExact;

// --- RTree + exact check ---
List<String> rtreeExact = new ArrayList<>();
var rt2 = jnr.ffi.Runtime.getSystemRuntime();
Pointer countPtr2 = jnr.ffi.Memory.allocate(rt2, Integer.BYTES);
long t0rtreeExact = System.currentTimeMillis();
for (ShipPos pos : shipPositions) {
    countPtr2.putInt(0, 0);
    Pointer idsPtr = meos.rtree_search(rtree, pos.box(), countPtr2);
    int count = countPtr2.getInt(0);
    if (idsPtr != null && count > 0) {
        int[] ids = new int[count];
        idsPtr.get(0, ids, 0, count);
        for (int id : ids) {
            PortBox port = portList.get(id);
            if (meos.geom_intersects2d(pos.geom(), port.geom())) {
                rtreeExact.add(String.format("MMSI %d @ %s -> %s (%s)",
                    pos.mmsi(), pos.ts(), port.portName(), port.portId()));
            }
        }
    }
}
long rtreeExactMs = System.currentTimeMillis() - t0rtreeExact;

var bruteExactSet = new java.util.TreeSet<>(bruteExact);
var rtreeExactSet = new java.util.TreeSet<>(rtreeExact);
boolean exactMatch = bruteExactSet.equals(rtreeExactSet);

System.out.println("+------------------------------------------+----------+----------+");
System.out.println("|                                          |  Brute   |  RTree   |");
System.out.println("+------------------------------------------+----------+----------+");
System.out.printf( "|  STBox only (section 4.4 / 4.5)          | %5d ms | %4d ms  |%n", bruteMs, rtreeMs);
System.out.printf( "|  STBox + geom_intersects2d (section 4.6) | %5d ms | %4d ms  |%n", bruteExactMs, rtreeExactMs);
System.out.println("+------------------------------------------+----------+----------+");
System.out.printf( "|  Matches (STBox only)                    | %8d | %8d |%n", bruteResults.size(), rtreeResults.size());
System.out.printf( "|  Matches (+ exact check)                 | %8d | %8d |%n", bruteExact.size(), rtreeExact.size());
System.out.printf( "|  False positives removed                 | %8d | %8d |%n",
    bruteResults.size() - bruteExact.size(),
    rtreeResults.size() - rtreeExact.size());
System.out.println("+------------------------------------------+----------+----------+");
System.out.printf( "|  RTree speedup (STBox only)              |     %.1fx faster     |%n",
    (double) bruteMs / Math.max(rtreeMs, 1));
System.out.printf( "|  RTree speedup (+ exact check)           |     %.1fx faster     |%n",
    (double) bruteExactMs / Math.max(rtreeExactMs, 1));
System.out.println("+------------------------------------------+----------+----------+");
System.out.println(exactMatch
    ? "|  Brute force and RTree: same exact results                     |"
    : "|  Results differ - check geom construction                       |");
System.out.println("+------------------------------------------+----------+----------+");

+------------------------------------------+----------+----------+
|                                          |  Brute   |  RTree   |
+------------------------------------------+----------+----------+
|  STBox only (section 4.4 / 4.5)          |   762 ms |  443 ms  |
|  STBox + geom_intersects2d (section 4.6) |   872 ms |  489 ms  |
+------------------------------------------+----------+----------+
|  Matches (STBox only)                    |    50938 |    50938 |
|  Matches (+ exact check)                 |    50317 |    50317 |
|  False positives removed                 |      621 |      621 |
+------------------------------------------+----------+----------+
|  RTree speedup (STBox only)              |     1.7x faster     |
|  RTree speedup (+ exact check)           |     1.8x faster     |
+------------------------------------------+----------+----------+
|  Brute force and RTree: same exact results                     |
+------------------------------------------+----------+-------

---
## Appendix - rapaio-jupyter-kernel constraints

The rapaio-jupyter-kernel executes cells as JShell snippets. Several behaviours differ from a standard Java class:

| Rule | Detail |
|---|---|
| `%jars` must be alone in its cell | Mixing it with comments or Java causes a failure |
| String returns from JNR-FFI crash the JVM | Declare as `Pointer`, read with `ptr.getString(0)` |
| Use `tgeompoint_in`, not `tgeogpoint_in` | The geography type requires a PROJ catalogue absent from standalone `libmeos.so` |
| You may find a formatter such as `Locale.US` useful in `String.format` | Without it, a comma decimal separator produces invalid WKT |
| `STBOX X((x y),(x y))` | Spatial-only boxes use double parentheses |
| `STBOX XT(((x y),(x y)),(t,t))` | Spatio-temporal boxes use triple parentheses |
| Kernel crash loses all state | Re-run all cells from `%jars` downward |